<a href="https://colab.research.google.com/github/ankurdev1-drth/Deep_Learning-/blob/main/02_Training_Deep_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [32]:
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [33]:
import torch

layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5  # kaiming init (or 3 ** 0.5 for LeCun)
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [34]:
layer.weight.shape

torch.Size([10, 40])

In [35]:
layer.bias.shape

torch.Size([10])

In [36]:
# another way to for initialization
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

This is clearer and less error prone as compared to the first approach

For applying initialization to the weights of every `nn.Linear` layer in a model the simples option is to write a little function that takes a module, checks whether it's an instance of the `nn.Linear` class, and if so , applies the desired initialization function to its weights. and the function can be then applied using the `apply()` method

In [37]:
def use_he_init(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_uniform_(module.weight)
    nn.init.zeros_(module.bias)

module = nn.Sequential(
    nn.Linear(40, 10),
    nn.ReLU(),
    nn.Linear(10, 1),
    nn.ReLU()
    )
module.apply(use_he_init)

Sequential(
  (0): Linear(in_features=40, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=1, bias=True)
  (3): ReLU()
)

### Leaky ReLU

In [38]:
alpha = 0.2
model = nn.Sequential(
    nn.Linear(50, 40), #0
    nn.LeakyReLU(negative_slope=alpha), #1
    nn.Linear(40, 20), #2
    nn.LeakyReLU(negative_slope=alpha) #3

)
nn.init.kaiming_uniform_(model[0].weight, alpha,
nonlinearity="leaky_relu")
nn.init.kaiming_uniform_(model[2].weight, alpha,nonlinearity="leaky_relu")

Parameter containing:
tensor([[ 2.7899e-01, -2.0753e-02, -1.0766e-01,  1.0289e-01, -1.7136e-01,
         -1.9297e-02,  1.1089e-01,  1.6970e-01, -3.6735e-01,  1.4706e-01,
         -1.4704e-01, -2.6750e-01, -1.6174e-01, -8.1937e-03,  1.8703e-01,
         -3.3824e-01,  1.0316e-01,  3.4217e-01, -2.1456e-01,  9.0750e-02,
         -1.5268e-01,  9.3369e-02,  1.6445e-01, -2.3395e-01,  9.0878e-02,
          3.7537e-01,  2.6646e-02, -2.6958e-01,  4.7504e-03,  8.3751e-02,
          1.1694e-01, -2.7378e-02, -8.1391e-02, -1.8883e-01,  2.4541e-01,
         -2.1454e-01, -1.2927e-01,  2.5198e-01, -1.7847e-01, -4.9569e-02],
        [-1.5142e-01, -3.3663e-01,  1.8307e-01, -1.6864e-01, -1.7525e-01,
         -8.3684e-02,  1.0405e-01,  3.3346e-01,  3.3592e-01,  5.1878e-03,
         -3.2572e-01,  2.8024e-02,  8.7757e-03,  3.2074e-01, -7.8497e-02,
         -3.5685e-01,  1.3797e-01, -8.3289e-02,  1.3252e-01,  1.1427e-01,
          1.3537e-01,  2.1007e-01, -2.6624e-01, -3.0934e-01,  3.5557e-01,
          3.471

### Batch Normalization

In [39]:
model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(1 * 28 * 28),
    nn.Linear(1 * 28 * 28,  300),
    nn.ReLU(),
    nn.BatchNorm1d(300),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.BatchNorm1d(100),
    nn.Linear(100, 10)
)

In [40]:
dict(model[1].named_parameters()).keys()

dict_keys(['weight', 'bias'])

In [41]:
dict(model[1].named_buffers()).keys()

dict_keys(['running_mean', 'running_var', 'num_batches_tracked'])

In [42]:
dict(model[1].named_children()).keys()

dict_keys([])

In [43]:
dict(model[1].named_modules()).keys()

dict_keys([''])

Note:
- if BN layers before the activation functions, we can also remove the bias term from the previous `nn.Linear` layers by setting the `bias` hyperparameter to `False`.

In [44]:
Model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 300, bias=False),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Linear(300, 100, bias=False),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

### Layer Normalization

In [45]:
inputs = torch.randn(32, 3, 100, 200)   # batch of random RGB images
layer_norm = nn.LayerNorm([100, 200])
result = layer_norm(inputs)
result

tensor([[[[-2.1608e-01, -1.6910e+00, -2.8689e-01,  ..., -3.6550e-01,
           -2.2364e-01, -7.6720e-05],
          [ 3.4315e-01,  2.8669e-01,  1.2245e-01,  ..., -7.0260e-01,
            2.1936e-01,  4.3340e-01],
          [ 2.9846e-02,  4.4110e-01, -1.4410e-01,  ...,  7.0779e-01,
            2.5174e+00,  8.7012e-01],
          ...,
          [ 1.3103e+00, -3.4669e-01,  8.5767e-01,  ..., -1.3561e-01,
           -2.0885e+00,  1.1085e-01],
          [-9.0293e-01,  9.5137e-01,  6.3664e-01,  ...,  1.6525e-01,
            9.7248e-01, -2.5192e-01],
          [ 1.7667e+00, -3.9028e-01,  1.1967e+00,  ..., -1.0014e+00,
           -5.7509e-01,  8.8270e-01]],

         [[-4.3885e-01,  1.8403e-01,  5.0075e-01,  ...,  8.1334e-01,
            3.7644e-01,  1.1345e+00],
          [ 1.8694e+00, -1.1136e+00, -2.7721e-01,  ...,  4.2443e-01,
           -1.2645e+00,  5.5062e-01],
          [-1.0898e+00,  2.8792e-01,  3.5266e-01,  ...,  7.3335e-01,
           -3.6326e-01,  1.6880e+00],
          ...,
     

In [46]:
# method 2 to perform the same thing as above!
means = inputs.mean(dim=[2,3], keepdim=True)  #shape [32, 3, 1, 1]
vars_ = inputs.var(dim=[2,3], keepdim=True, unbiased=False) # shape: same
stds = torch.sqrt(vars_ + layer_norm.eps) # eps is a smoothing term (1e-5)
result = layer_norm.weight * (inputs - means) / stds + layer_norm.bias

In [47]:
layer_norm = nn.LayerNorm([3, 100, 200])
result = layer_norm(inputs)

In [48]:
result

tensor([[[[-0.2124, -1.6867, -0.2832,  ..., -0.3618, -0.2199,  0.0035],
          [ 0.3466,  0.2902,  0.1260,  ..., -0.6987,  0.2229,  0.4368],
          [ 0.0334,  0.4445, -0.1404,  ...,  0.7111,  2.5200,  0.8734],
          ...,
          [ 1.3134, -0.3430,  0.8609,  ..., -0.1320, -2.0841,  0.1144],
          [-0.8990,  0.9546,  0.6400,  ...,  0.1688,  0.9757, -0.2482],
          [ 1.7696, -0.3865,  1.1999,  ..., -0.9974, -0.5713,  0.8860]],

         [[-0.4360,  0.1824,  0.4968,  ...,  0.8072,  0.3734,  1.1260],
          [ 1.8555, -1.1059, -0.2755,  ...,  0.4211, -1.2557,  0.5463],
          [-1.0822,  0.2855,  0.3498,  ...,  0.7277, -0.3610,  1.6755],
          ...,
          [-0.0597, -0.0722, -1.6290,  ..., -0.3516,  1.5151, -0.2346],
          [ 0.5490,  0.7447,  0.2642,  ..., -1.3541,  0.0599, -0.8363],
          [-1.1646,  0.6088,  0.6826,  ...,  0.1261, -0.6490, -0.9208]],

         [[-1.2884, -0.4363, -0.3195,  ..., -0.0641, -1.3135,  0.2823],
          [ 0.4279,  0.5472,  

## Gradient Clipping

See the line nn.utils.clip_grad_norm_(...) in the training function in the next section.

# Reusing Pretrained Layers

### Transfer Learning with PyTorch

- X_train_A = images of all items except for T-shirts/tops and pullovers (classes 0 and 2 )
- X_train_B = first 20 images of Tshirt and Pullovers

The validation set and the test set are also split this way, but without restricting the number of images.

We will train a model on set A (classification task with 8 classes), and try to reuse it to tackle set B (binary classification). We hope to transfer a little bit of knowledge from task A to task B, since classes in set A (trousers, dresses, coats, sandals, shirts, sneakers, bags, and ankle boots) are somewhat similar to classes in set B (T-shirts/tops and pullovers). However, since we are using Linear layers, only patterns that occur at the same location can be reused (in contrast, convolutional layers will transfer much better, since learned patterns can be detected anywhere on the image)


In [49]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_openml


fashion_mnist = fetch_openml(name="Fashion-MNIST", as_frame=False) #as_frames = False for getting the numpy arrays instead of pandas dataframe/series
X = torch.FloatTensor(fashion_mnist.data.reshape(-1, 1, 28, 28) / 255.)
y = torch.from_numpy(fashion_mnist.target.astype(int))
in_B = (y == 0) | (y == 2)  # Pullover or T-shirt/top
X_A, y_A = X[~in_B], y[~in_B]
y_A = torch.maximum(y_A - 2, torch.tensor(0))  # [1,3,4,5,6,7,8,9] => [0,..,7]
X_B, y_B = X[in_B], (y[in_B] == 2).to(dtype=torch.float32).view(-1, 1)

train_set_A = TensorDataset(X_A[:-7_000], y_A[:-7000])
valid_set_A = TensorDataset(X_A[-7_000:-5000], y_A[-7000:-5000])
test_set_A = TensorDataset(X_A[-5_000:], y_A[-5000:])
train_set_B = TensorDataset(X_B[:20], y_B[:20])
valid_set_B = TensorDataset(X_B[20:5000], y_B[20:5000])
test_set_B = TensorDataset(X_B[5_000:], y_B[5000:])

train_loader_A = DataLoader(train_set_A, batch_size=32, shuffle=True)
valid_loader_A = DataLoader(valid_set_A, batch_size=32)
test_loader_A = DataLoader(test_set_A, batch_size=32)
train_loader_B = DataLoader(train_set_B, batch_size=32, shuffle=True)
valid_loader_B = DataLoader(valid_set_B, batch_size=32)
test_loader_B = DataLoader(test_set_B, batch_size=32)

In [50]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device


'cpu'

In [51]:
%pip install torchmetrics


import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            # Uncomment to activate gradient clipping:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history



In [52]:
torch.manual_seed(42)

model_A = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 8)
  )
model_A = model_A.to(device)

In [53]:
model_A.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): ReLU()
  (7): Linear(in_features=100, out_features=8, bias=True)
)

In [54]:
n_epochs = 20
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.005)
loss_fn = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=8).to(device)
history_A = train(model_A, optimizer, loss_fn, accuracy, train_loader_A, valid_loader_A, n_epochs )

Epoch 1/20, train loss: 0.8492, train metric: 0.7197, valid metric: 0.8120
Epoch 2/20, train loss: 0.4533, train metric: 0.8487, valid metric: 0.8620
Epoch 3/20, train loss: 0.3771, train metric: 0.8727, valid metric: 0.8685
Epoch 4/20, train loss: 0.3405, train metric: 0.8827, valid metric: 0.8890
Epoch 5/20, train loss: 0.3184, train metric: 0.8904, valid metric: 0.8825
Epoch 6/20, train loss: 0.3032, train metric: 0.8953, valid metric: 0.8880
Epoch 7/20, train loss: 0.2911, train metric: 0.8999, valid metric: 0.8910
Epoch 8/20, train loss: 0.2817, train metric: 0.9031, valid metric: 0.8960
Epoch 9/20, train loss: 0.2735, train metric: 0.9053, valid metric: 0.8985
Epoch 10/20, train loss: 0.2666, train metric: 0.9083, valid metric: 0.8900
Epoch 11/20, train loss: 0.2610, train metric: 0.9108, valid metric: 0.9025
Epoch 12/20, train loss: 0.2546, train metric: 0.9127, valid metric: 0.8985
Epoch 13/20, train loss: 0.2502, train metric: 0.9146, valid metric: 0.8860
Epoch 14/20, train lo

In [55]:
torch.manual_seed(42)
model_B = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 1)
).to(device)

In [56]:
model_B.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): ReLU()
  (7): Linear(in_features=100, out_features=1, bias=True)
)

In [57]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B, optimizer, loss_fn, accuracy, train_loader_B, valid_loader_B, n_epochs)

Epoch 1/20, train loss: 0.7110, train metric: 0.5000, valid metric: 0.4914
Epoch 2/20, train loss: 0.7083, train metric: 0.5000, valid metric: 0.4954
Epoch 3/20, train loss: 0.7055, train metric: 0.5000, valid metric: 0.4976
Epoch 4/20, train loss: 0.7028, train metric: 0.5000, valid metric: 0.5008
Epoch 5/20, train loss: 0.7001, train metric: 0.5000, valid metric: 0.5034
Epoch 6/20, train loss: 0.6974, train metric: 0.5000, valid metric: 0.5080
Epoch 7/20, train loss: 0.6947, train metric: 0.5000, valid metric: 0.5118
Epoch 8/20, train loss: 0.6920, train metric: 0.5000, valid metric: 0.5159
Epoch 9/20, train loss: 0.6893, train metric: 0.5000, valid metric: 0.5197
Epoch 10/20, train loss: 0.6866, train metric: 0.5000, valid metric: 0.5225
Epoch 11/20, train loss: 0.6840, train metric: 0.5000, valid metric: 0.5263
Epoch 12/20, train loss: 0.6813, train metric: 0.5000, valid metric: 0.5311
Epoch 13/20, train loss: 0.6787, train metric: 0.5000, valid metric: 0.5339
Epoch 14/20, train lo

In [58]:
# reusing all the layers except the output layer
import copy

torch.manual_seed(42)
reused_layer = copy.deepcopy(model_A[: -1])
model_B_on_A = nn.Sequential(
    *reused_layer,
    nn.Linear(100, 1) # new output layer for task B
).to(device)

Note:
- `copy.deepcopy()` function to copy all the modules and submodules in `nn.Sequential` module
- model_B_on_A is another `nn.Sequential` module based on the reused layers of model A plus a new output layer for task B it has a single output since task B is binary classification.

In [59]:
# freezing the reused layers during the first few epochs:
for layer in model_B_on_A[:-1]:
  for param in layer.parameters():
    param.requires_grad = False


In [60]:
n_epochs = 10
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs)

Epoch 1/10, train loss: 0.6172, train metric: 0.4000, valid metric: 0.5378
Epoch 2/10, train loss: 0.5871, train metric: 0.4500, valid metric: 0.5759
Epoch 3/10, train loss: 0.5609, train metric: 0.6500, valid metric: 0.6373
Epoch 4/10, train loss: 0.5385, train metric: 0.8000, valid metric: 0.7124
Epoch 5/10, train loss: 0.5201, train metric: 0.8500, valid metric: 0.7817
Epoch 6/10, train loss: 0.5055, train metric: 0.8500, valid metric: 0.8327
Epoch 7/10, train loss: 0.4945, train metric: 0.9000, valid metric: 0.8639
Epoch 8/10, train loss: 0.4863, train metric: 0.9000, valid metric: 0.8791
Epoch 9/10, train loss: 0.4794, train metric: 0.8500, valid metric: 0.8851
Epoch 10/10, train loss: 0.4729, train metric: 0.8500, valid metric: 0.8888


In [61]:
# unfreezing the reused layers during the first few epochs:
for layer in model_B_on_A[2:]:
  for param in layer.parameters():
    param.requires_grad = True

In [64]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.01)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                train_loader_B, valid_loader_B, n_epochs)

Epoch 1/20, train loss: 0.1987, train metric: 1.0000, valid metric: 0.9365
Epoch 2/20, train loss: 0.1881, train metric: 1.0000, valid metric: 0.9384
Epoch 3/20, train loss: 0.1845, train metric: 1.0000, valid metric: 0.9384
Epoch 4/20, train loss: 0.1818, train metric: 1.0000, valid metric: 0.9384
Epoch 5/20, train loss: 0.1791, train metric: 1.0000, valid metric: 0.9384
Epoch 6/20, train loss: 0.1766, train metric: 1.0000, valid metric: 0.9388
Epoch 7/20, train loss: 0.1741, train metric: 1.0000, valid metric: 0.9390
Epoch 8/20, train loss: 0.1717, train metric: 1.0000, valid metric: 0.9388
Epoch 9/20, train loss: 0.1694, train metric: 1.0000, valid metric: 0.9390
Epoch 10/20, train loss: 0.1671, train metric: 1.0000, valid metric: 0.9390
Epoch 11/20, train loss: 0.1649, train metric: 1.0000, valid metric: 0.9392
Epoch 12/20, train loss: 0.1628, train metric: 1.0000, valid metric: 0.9398
Epoch 13/20, train loss: 0.1607, train metric: 1.0000, valid metric: 0.9396
Epoch 14/20, train lo

In [65]:
evaluate_tm(model_B_on_A, test_loader_B, accuracy)

tensor(0.9422)

## Faster Optimizers

In [66]:
# to test an optimizer on fashion MNIST :
train_set = TensorDataset(X[:55_000], y[:55_000])
valid_set = TensorDataset(X[55_000:60_000], y[55_000:60_000])
test_set = TensorDataset(X[60_000:], y[60_000:])

def build_model(seed=43):
  torch.manual_seed(seed)
  model = nn.Sequential(
      nn.Flatten(),
      nn.Linear(28 * 28, 100),
      nn.ReLU(),
      nn.Linear(100, 100),
      nn.ReLU(),
      nn.Linear(100, 100),
      nn.ReLU(),
      nn.Linear(100, 10)
  ).to(device)
  model.apply(use_he_init)
  return model

def test_optimizer(model, optimizer, n_epochs=10, batch_size=32):
  train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
  valid_loader = DataLoader(valid_set, batch_size=batch_size)
  test_loader = DataLoader(test_set, batch_size=32)
  xentropy = nn.CrossEntropyLoss()
  accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
  history = train(model, optimizer, xentropy, accuracy.to(device),
                  train_loader, valid_loader, test_loader)
  return history, evaluate_tm(model, test_loader, accuracy)
